# 08: Group-aware sampling and shortcut learning

![Group-aware pipeline](../images/08_group_aware_sampling.svg)

**Learning goals:** split independent groups, construct fixed 16-frame anchor support, compare frozen-random and resampled policies, pair nuisance streams, audit expected realized support at fixed exposure, record lineage, and measure shortcut inflation. All data are synthetic.

In [ ]:
import hashlib
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

SEED = 8
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print(f'NumPy {np.__version__}, synthetic data only')

## 1. Build repeated observations with acquisition artifacts

Each group receives a random binary label and a strong group-specific artifact vector. Repeated rows from a group share that artifact. The final coordinate contains a weak intended label signal. A nearest-neighbor model can exploit group identity whenever the same groups occur in training and test data.

In [ ]:
n_groups, repeats, artifact_dim = 60, 8, 10
group_labels = rng.integers(0, 2, size=n_groups)
artifacts = rng.normal(0, 4.0, size=(n_groups, artifact_dim))
groups = np.repeat(np.arange(n_groups), repeats)
labels = group_labels[groups]
artifact_features = artifacts[groups] + rng.normal(0, 0.15, size=(len(groups), artifact_dim))
intended = (2 * labels - 1)[:, None] * 0.25 + rng.normal(0, 1.0, size=(len(groups), 1))
features = np.concatenate([artifact_features, intended], axis=1)
print(f'features={features.shape}, groups={np.unique(groups).size}, rows/group={repeats}')
assert features.shape == (n_groups * repeats, artifact_dim + 1)

## 2. Compare a row split with a group-disjoint split

`train_test_split` knows nothing about groups. `GroupShuffleSplit` assigns all rows with one group key together. The test below asserts the defining invariant: the two group sets have an empty intersection.

In [ ]:
indices = np.arange(len(features))
row_train, row_test = train_test_split(indices, test_size=0.3, random_state=SEED, stratify=labels)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
group_train, group_test = next(splitter.split(features, labels, groups=groups))

row_overlap = set(groups[row_train]) & set(groups[row_test])
group_overlap = set(groups[group_train]) & set(groups[group_test])
print(f'row split shared groups={len(row_overlap)}; group split shared groups={len(group_overlap)}')
assert len(group_overlap) == 0
assert len(row_overlap) > 0

In [ ]:
def fit_score(train_idx, test_idx, use_artifacts=True):
    columns = slice(None) if use_artifacts else [-1]
    model = KNeighborsClassifier(n_neighbors=1)
    model.fit(features[train_idx][:, columns], labels[train_idx])
    return accuracy_score(labels[test_idx], model.predict(features[test_idx][:, columns]))

scores = {
    'row split, all features': fit_score(row_train, row_test),
    'group split, all features': fit_score(group_train, group_test),
    'group split, intended only': fit_score(group_train, group_test, use_artifacts=False),
}
for name, score in scores.items():
    print(f'{name:30s}: {score:.3f}')
assert scores['row split, all features'] > scores['group split, all features'] + 0.2

The score gap is the lesson, not a universal numerical threshold. The row split rewards recognition of group artifacts. Group-disjoint evaluation asks whether the rule transfers to unseen groups. An artifact-only baseline, subgroup scores, and counterfactual artifact changes are complementary diagnostics.

## 3. Build the fixed temporal anchor inventory

For $n_i$ frames and clip length $T=16$, there are $W_i=n_i-T+1$ valid starts. The intervention uses $\mathcal A_i=\{0,8,16,\ldots,8\lfloor(W_i-1)/8\rfloor\}$ and $K_i=|\mathcal A_i|$. Sequences must have $K_i\ge2$ before any pool is built. Adjacent windows overlap by 8 of 16 frames, or 50 percent. A 16-frame grid gives a conservative non-overlapping count.

In [ ]:
CLIP_LENGTH = 16
ANCHOR_STRIDE = 8

def anchor_starts(n_frames, stride=ANCHOR_STRIDE):
    if n_frames < CLIP_LENGTH or stride <= 0:
        raise ValueError('sequence is too short or stride is not positive')
    valid_start_count = n_frames - CLIP_LENGTH + 1
    return np.arange(0, valid_start_count, stride, dtype=int)

def overlap_frames(start_a, start_b):
    return max(0, CLIP_LENGTH - abs(int(start_a) - int(start_b)))

anchors = anchor_starts(48)
nonoverlap = anchor_starts(48, stride=CLIP_LENGTH)
assert anchors.tolist() == [0, 8, 16, 24, 32]
assert len(anchors) == 5 and nonoverlap.tolist() == [0, 16, 32]
assert overlap_frames(anchors[0], anchors[1]) / CLIP_LENGTH == 0.5
assert overlap_frames(nonoverlap[0], nonoverlap[1]) == 0
assert len(anchor_starts(24)) == 2 and len(anchor_starts(23)) == 1
for n_frames in range(CLIP_LENGTH, 200):
    starts = anchor_starts(n_frames)
    valid_start_count = n_frames - CLIP_LENGTH + 1
    assert len(starts) == (valid_start_count - 1) // ANCHOR_STRIDE + 1
    assert np.all(starts + CLIP_LENGTH <= n_frames)
print(f'W_i={48 - CLIP_LENGTH + 1}, K_i={len(anchors)}, conservative count={len(nonoverlap)}')

## 4. Assign policies with stable identities and named streams

Frozen-random chooses one uniform anchor for each `(sequence_id, replicate_seed)` pair and repeats it within that run. Resampling makes a uniform choice on every draw. Both are uniform across randomization, but only resampling can realize several anchors for one sequence. A versioned stable hash makes choices independent of Python process state and manifest row position. Separate sequence, temporal, spatial, and mask streams keep nuisance draws paired.

In [ ]:
SEQUENCE_STREAM = 'sequence-v1'
FROZEN_STREAM = 'temporal-frozen-v1'
RESAMPLED_STREAM = 'temporal-resampled-v1'
SPATIAL_STREAM = 'spatial-v1'
MASK_STREAM = 'mask-v1'

def stable_uint64(namespace, *parts):
    payload = json.dumps([namespace, *parts], separators=(',', ':'), ensure_ascii=False).encode()
    digest = hashlib.blake2b(payload, digest_size=8).digest()
    return int.from_bytes(digest, 'little')

def frozen_anchor(sequence_id, replicate_seed, supported_anchors):
    local = np.random.default_rng(stable_uint64(FROZEN_STREAM, sequence_id, replicate_seed))
    return int(local.choice(supported_anchors))

def resampled_anchor(sequence_id, replicate_seed, virtual_epoch, draw_index, supported_anchors):
    seed = stable_uint64(RESAMPLED_STREAM, replicate_seed, virtual_epoch, sequence_id, draw_index)
    return int(np.random.default_rng(seed).choice(supported_anchors))

sequence_lengths = {f'seq-{i:02d}': 48 + 8 * i for i in range(8)}
anchor_inventory = {key: anchor_starts(length) for key, length in sequence_lengths.items()}
high_manifest = list(sequence_lengths)
low_manifest = high_manifest[:4]
assert len(low_manifest) < len(high_manifest)
assert high_manifest[:len(low_manifest)] == low_manifest
assert set(low_manifest) < set(high_manifest)
frozen_low = {key: frozen_anchor(key, SEED, anchor_inventory[key]) for key in low_manifest}
frozen_high = {key: frozen_anchor(key, SEED, anchor_inventory[key]) for key in high_manifest}
assert all(frozen_low[key] == frozen_high[key] for key in low_manifest)

probe = np.array([0, 8, 16, 24])
frozen_probe = np.array([frozen_anchor('probe', replicate, probe) for replicate in range(8_000)])
resampled_probe = np.array([resampled_anchor('probe', SEED, 0, draw, probe) for draw in range(8_000)])
for draws in (frozen_probe, resampled_probe):
    frequencies = np.array([(draws == anchor).mean() for anchor in probe])
    assert np.max(np.abs(frequencies - 0.25)) < 0.03
assert len({frozen_anchor('probe', SEED, probe) for _ in range(20)}) == 1

def draw_record(manifest, policy, replicate_seed, virtual_epoch, draw_index):
    sequence_rng = np.random.default_rng(
        stable_uint64(SEQUENCE_STREAM, replicate_seed, virtual_epoch, draw_index))
    sequence_id = str(sequence_rng.choice(manifest))
    supported = anchor_inventory[sequence_id]
    if policy == 'frozen_random':
        start = frozen_anchor(sequence_id, replicate_seed, supported)
    elif policy == 'resampled_anchor':
        start = resampled_anchor(sequence_id, replicate_seed, virtual_epoch, draw_index, supported)
    else:
        raise ValueError('unknown temporal policy')
    spatial_rng = np.random.default_rng(
        stable_uint64(SPATIAL_STREAM, replicate_seed, virtual_epoch, sequence_id, draw_index))
    mask_rng = np.random.default_rng(
        stable_uint64(MASK_STREAM, replicate_seed, virtual_epoch, sequence_id, draw_index))
    spatial = tuple(spatial_rng.integers(0, 5, size=2).tolist())
    mask = tuple(sorted(mask_rng.choice(16, size=4, replace=False).tolist()))
    return {'sequence_id': sequence_id, 'start': start, 'spatial': spatial, 'mask': mask}

frozen_records = [draw_record(low_manifest, 'frozen_random', SEED, 0, draw) for draw in range(64)]
resampled_records = [draw_record(low_manifest, 'resampled_anchor', SEED, 0, draw) for draw in range(64)]
assert len(frozen_records) == len(resampled_records) == 64
assert all(a['sequence_id'] == b['sequence_id'] for a, b in zip(frozen_records, resampled_records))
assert all(a['spatial'] == b['spatial'] and a['mask'] == b['mask']
           for a, b in zip(frozen_records, resampled_records))
assert any(a['start'] != b['start'] for a, b in zip(frozen_records, resampled_records))
print('nested anchors stable; marginal uniformity and paired nuisance streams verified')

## 5. Audit expected realized support before training

For $U$ uniformly sampled sequences and exposure $C$, frozen expected support is $E_F=U[1-(1-1/U)^C]$. Resampled expected support is $E_R=\sum_i K_i[1-(1-1/(UK_i))^C]$. These occupancy expectations count visited sequence-anchor pairs, not independent examples. The prospective gates require median $K_i\ge4$ and $E_R/E_F\ge4$ in every nested pool at the lower exposure of 4,096,000 examples.

In [ ]:
PROSPECTIVE_EXPOSURE = 4_096_000

def probability_visited(draw_probability, exposure):
    with np.errstate(divide='ignore', invalid='ignore'):
        return float(-np.expm1(exposure * np.log1p(-draw_probability)))

def expected_frozen_support(pool_size, exposure):
    return pool_size * probability_visited(1.0 / pool_size, exposure)

def expected_resampled_support(anchor_counts, exposure):
    pool_size = len(anchor_counts)
    return sum(k * probability_visited(1.0 / (pool_size * k), exposure)
               for k in anchor_counts)

def repeated_draw_fraction(expected_distinct_pairs, exposure):
    return 1.0 - expected_distinct_pairs / exposure

def mean_distinct_anchor_overlap(manifest):
    overlap_fractions = []
    for sequence_id in manifest:
        supported = anchor_inventory[sequence_id]
        for left in range(len(supported)):
            for right in range(left + 1, len(supported)):
                overlap_fractions.append(
                    overlap_frames(supported[left], supported[right]) / CLIP_LENGTH)
    if not overlap_fractions:
        raise ValueError('at least one distinct anchor pair is required')
    return float(np.mean(overlap_fractions))

audit_rows = []
for pool_name, manifest in [('low', low_manifest), ('high', high_manifest)]:
    k = np.array([len(anchor_inventory[key]) for key in manifest])
    q = np.array([len(anchor_starts(sequence_lengths[key], CLIP_LENGTH)) for key in manifest])
    expected_f = expected_frozen_support(len(manifest), PROSPECTIVE_EXPOSURE)
    expected_r = expected_resampled_support(k, PROSPECTIVE_EXPOSURE)
    ratio = expected_r / expected_f
    frozen_repeat_fraction = repeated_draw_fraction(expected_f, PROSPECTIVE_EXPOSURE)
    resampled_repeat_fraction = repeated_draw_fraction(expected_r, PROSPECTIVE_EXPOSURE)
    mean_overlap = mean_distinct_anchor_overlap(manifest)
    audit_rows.append({
        'pool': pool_name, 'median_k': float(np.median(k)),
        'median_nonoverlap': float(np.median(q)), 'expected_f': expected_f,
        'expected_r': expected_r, 'support_ratio': ratio,
        'frozen_repeated_draw_fraction': frozen_repeat_fraction,
        'resampled_repeated_draw_fraction': resampled_repeat_fraction,
        'mean_distinct_anchor_overlap': mean_overlap,
    })
    assert np.median(k) >= 4
    assert ratio >= 4
    assert expected_f <= len(manifest) and expected_r <= k.sum()
    assert 0 <= resampled_repeat_fraction <= frozen_repeat_fraction <= 1
    assert 0 <= mean_overlap <= 0.5

manifest_digest = hashlib.sha256('\n'.join(high_manifest).encode()).hexdigest()
example = resampled_records[0]
lineage = {
    **example, 'stop': example['start'] + CLIP_LENGTH, 'replicate_seed': SEED,
    'temporal_policy': 'resampled_anchor', 'temporal_stream': RESAMPLED_STREAM,
    'sequence_stream': SEQUENCE_STREAM, 'spatial_stream': SPATIAL_STREAM,
    'mask_stream': MASK_STREAM, 'manifest_digest': manifest_digest,
    'planned_exposure': PROSPECTIVE_EXPOSURE, 'preprocess_version': 'synthetic-v1',
}
assert lineage['start'] in anchor_inventory[lineage['sequence_id']]
assert lineage['stop'] - lineage['start'] == CLIP_LENGTH
assert len(lineage['manifest_digest']) == 64
required_audit_fields = {
    'pool', 'median_k', 'median_nonoverlap', 'expected_f', 'expected_r',
    'support_ratio', 'frozen_repeated_draw_fraction',
    'resampled_repeated_draw_fraction', 'mean_distinct_anchor_overlap',
}
assert all(set(row) == required_audit_fields for row in audit_rows)
for row in audit_rows:
    print(f"{row['pool']}: median K={row['median_k']:.1f}, "
          f"E_F={row['expected_f']:.1f}, E_R={row['expected_r']:.1f}, "
          f"ratio={row['support_ratio']:.2f}, "
          f"repeat R={row['resampled_repeated_draw_fraction']:.6f}, "
          f"mean overlap={row['mean_distinct_anchor_overlap']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.8), constrained_layout=True)
ax.barh(list(scores), list(scores.values()), color=['#c75b4b', '#367cad', '#27896f'])
ax.set(xlim=(0, 1), xlabel='accuracy', title='A row split rewards the group shortcut')
ax.axvline(0.5, color='black', linestyle=':', linewidth=1)
plt.show()

## Exercises and takeaways

1. Change a sequence length and recompute $W_i$, $K_i$, and the conservative non-overlapping count. **Check:** every anchor still produces exactly 16 frames.
2. Remove the stream namespace from `stable_uint64`. **Check:** unrelated random choices can collide or become coupled.
3. Increase the synthetic pool size and lower exposure. **Check:** expected realized support can fall below available support.
4. Group by each row instead of the true group. **Check:** formal disjointness no longer protects the independent unit.

**Efficiency:** `np.isin` forms group masks without row loops, index manifests avoid storing duplicate windows, and view-based window functions save memory if callers avoid unsafe in-place writes.

**Takeaways:** split independent groups before creating windows; define fixed temporal support; use stable identity hashing; keep frozen and resampled marginals equal; pair sequence, spatial, and mask streams; hold exposure fixed; audit occupancy gates prospectively; and retain lineage so every tensor can be reconstructed.

## Continue learning

[Previous notebook: 07](07_gradient_updates_and_schedules.ipynb) | [Lecture](../lectures/08_group_aware_sampling.md) | [Curriculum](../README.md) | [Next notebook: 09](09_eigenspectra_and_effective_rank.ipynb)